# Executor de experimentos

- Usado para a execução de cada experimento no ambiente do Google Colab.
- Cada experimento foi realizado em três rodadas com seeds diferentes
- Os resultados de cada experimento foram salvas em arquivos no Google Drive e num banco de dados

Verificação do ambiente de execução

In [ ]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))

!nvidia-smi

Clonagem do repositório atualizado no GitHub

In [ ]:
%cd /content
!git clone https://github.com/amartinsmg/classification-of-medical-images-using-cnn.git
%cd /content/classification-of-medical-images-using-cnn

Montagem do Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
BASE_PATH = "/content/drive/MyDrive/classification-of-medical-images-using-cnn/"

Configuração dos parâmetros do experimento

In [ ]:
EXPERIMENT_NAME = "efficientnet-rescaling"
BASE_MODEL = "efficientnet"
NORMALIZATION = "rescaling"
DATA_AUG = True
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_WEIGHT = False
LEARNING_RATE = 0.001
EPOCHS = 10
THRESHOLD = 0.5
seeds = [42, 123, 999]

Criação da engine do banco de dados
- Foi usado um banco de dados sqlite hospedado também no Google Drive

In [ ]:
from src.db import insert_run, get_engine

DB_PATH = f"sqlite:///{BASE_PATH}/experiments.db"

engine = get_engine(DB_PATH)

Realização das três rodadas do experimento com:
- Treinamento do modelo
- Teste do modelo treinado
- Persitência dos parâmetros e dos resulados da rodada em banco de dados

In [ ]:
from src.train import train_pipeline
from src.test import test_pipeline

for i in range(3):
    run_id = i + 1
    config = train_pipeline(
        base_dir=BASE_PATH,
        experiment_name=EXPERIMENT_NAME,
        run_id=run_id,
        normalization=NORMALIZATION,
        base_model=BASE_MODEL,
        data_augmentation=DATA_AUG,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        class_weights=CLASS_WEIGHT,
        learning_rate=LEARNING_RATE,
        epochs=EPOCHS,
        seed=seeds[i],
    )

    metrics = test_pipeline(
        base_dir=BASE_PATH,
        experiment_name=EXPERIMENT_NAME,
        run_id=run_id,
        normalization=NORMALIZATION,
        base_model=BASE_MODEL,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        threshold=THRESHOLD,
    )

    insert_run(
        engine=engine,
        experiment=EXPERIMENT_NAME,
        run_name=f"run-{run_id}",
        config=config,
        metrics=metrics,
    )

Instala DagsHub para versionamento dos resultados dos experimentos
- O DagsHub é uma plataforma de versionamento de dados e modelos, similar ao GitHub, mas focada em projetos de ciência de dados e machine learning

In [ ]:
%pip install -q dvc dagshub

Loga no DagsHub com Secrets do Google Colab

In [ ]:
import dagshub
from google.colab import userdata

dagshub.auth.add_app_token(token=userdata.get("DAGSHUB_TOKEN"))

Faz upload da pasta com os resultados do último experimento feito

In [ ]:
RESULTS_RELATIVE_PATH = "results/" + EXPERIMENT_NAME
LOCAL_PATH = BASE_PATH + "/" + RESULTS_RELATIVE_PATH

dagshub.upload_files(
    "amartinsmg/classification-of-medical-images-using-cnn",
    local_path=LOCAL_PATH,
    remote_path=RESULTS_RELATIVE_PATH,
)